# 07 — Telugu ASR: wav2vec2 Learning Curve Study

**Research question:** How does training data size affect Telugu ASR performance?

Run this notebook four times, changing only `DATA_FRACTION` each time:

| Run | DATA_FRACTION | DATASET_SOURCE | Approx. hours |
|-----|--------------|----------------|---------------|
| 1   | 0.50         | mscil          | ~45h          |
| 2   | 0.10         | mscil          | ~9h           |
| 3   | 0.25         | mscil          | ~22h          |
| 4   | 1.00         | mscil          | ~90h          |
| 5   | 1.00         | multi          | ~190h         |

Results for each run are saved to `./results/wav2vec2_lc_{source}_{fraction}.json`  
so they can be compared in the analysis notebook.

**Evaluation benchmark:** FLEURS te_in test set (held-out — never used for training)

In [ ]:
!pip install -q \
    "typing_extensions>=4.6" \
    "transformers>=4.46" \
    "datasets<3.0" \
    "accelerate>=0.26" \
    "evaluate" \
    "jiwer" \
    "soundfile" \
    "librosa" \
    "pyctcdecode" \
    "kenlm"
print("Done.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CHANGE THESE TWO LINES BETWEEN RUNS
DATA_FRACTION  = 0.50    # 0.10 | 0.25 | 0.50 | 1.00
DATASET_SOURCE = "mscil" # "mscil" | "multi"
# ═══════════════════════════════════════════════════════════

import os, re, json, logging, subprocess, sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import torch
import evaluate
from datasets import load_from_disk, load_dataset, Audio
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from huggingface_hub import login

logging.getLogger("numba").setLevel(logging.WARNING)

_hf_token = os.environ.get("HF_TOKEN")
if _hf_token:
    login(token=_hf_token)

# ── Paths ────────────────────────────────────────────────────
_SOURCE_PATHS = {
    "mscil": "./telugu_asr_clean_dataset",
    "multi": "./telugu_multi_dataset",
}
_PROCESSOR_PATH = (
    "./telugu_wav2vec2_processor_v2"
    if Path("./telugu_wav2vec2_processor_v2").exists()
    else "./telugu_wav2vec2_processor"
)
DATASET_PATH = _SOURCE_PATHS[DATASET_SOURCE]
assert Path(DATASET_PATH).exists(), f"Dataset not found: {DATASET_PATH}"

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
RUN_TAG    = f"{DATASET_SOURCE}_{DATA_FRACTION}"
OUTPUT_DIR = f"./wav2vec2-lc-{RUN_TAG}"
RESULTS_DIR = "./results"
Path(RESULTS_DIR).mkdir(exist_ok=True)

# ── EXP-1 hyperparameters (same as notebook 05) ───────────────
LR                   = 1e-4
WARMUP               = 1000
EPOCHS               = 50
EVAL_STEPS           = 500
SAVE_STEPS           = 500
SEED                 = 42
FP16                 = True
MASK_TIME_PROB       = 0.1
MASK_FEAT_PROB       = 0.1
ATTENTION_DROPOUT    = 0.1
HIDDEN_DROPOUT       = 0.1
FEAT_PROJ_DROPOUT    = 0.0
EARLY_STOP_PATIENCE  = 8
EARLY_STOP_THRESHOLD = 0.005

# ── Memory-adaptive batch sizing (total VRAM — assumes this is
#    the only active training job on the GPU) ──────────────────
assert torch.cuda.is_available(), "No GPU found."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 40:    BATCH_SIZE, GRAD_ACCUM = 8, 4
elif vram_gb >= 20:  BATCH_SIZE, GRAD_ACCUM = 4, 8
elif vram_gb >= 12:  BATCH_SIZE, GRAD_ACCUM = 2, 16
else:                BATCH_SIZE, GRAD_ACCUM = 1, 32

torch.backends.cuda.matmul.allow_tf32 = True
print(f"GPU        : {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB)")
print(f"Run tag    : {RUN_TAG}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Fraction   : {DATA_FRACTION}")
print(f"Batch      : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective")

In [ ]:
# ── Load processor ──────────────────────────────────────────────
processor = Wav2Vec2Processor.from_pretrained(_PROCESSOR_PATH)
print(f"Processor vocab size: {processor.tokenizer.vocab_size}")

# ── Load training corpus ────────────────────────────────────────
raw_ds = load_from_disk(DATASET_PATH)
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=16_000))

# Subsample by DATA_FRACTION (stratified shuffle for reproducibility)
if DATA_FRACTION < 1.0:
    n_keep = int(len(raw_ds) * DATA_FRACTION)
    raw_ds = raw_ds.shuffle(seed=SEED).select(range(n_keep))
    print(f"Subsampled to {len(raw_ds):,} samples ({DATA_FRACTION*100:.0f}%)")

split    = raw_ds.train_test_split(test_size=0.1, seed=SEED)
train_ds = split["train"]
val_ds   = split["test"]   # internal validation (for early stopping)

# ── Load FLEURS te_in test set — held-out benchmark ─────────────
fleurs_test = None
try:
    _fl = load_dataset("google/fleurs", "te_in", trust_remote_code=True)
    fleurs_test = _fl["test"].select_columns(["audio", "transcription"])
    fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=16_000))
    print(f"FLEURS te_in test  : {len(fleurs_test):,} samples (benchmark)")
except Exception as e:
    print(f"[WARN] FLEURS unavailable: {e}")

print(f"Train : {len(train_ds):,}")
print(f"Val   : {len(val_ds):,}")

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

KEEP = {"input_values", "input_length", "labels"}

print("Preparing train features...")
train_prep = train_ds.map(
    prepare_dataset,
    remove_columns=[c for c in train_ds.column_names if c not in KEEP],
    num_proc=4, desc="train",
).sort("input_length")

print("Preparing val features...")
val_prep = val_ds.map(
    prepare_dataset,
    remove_columns=[c for c in val_ds.column_names if c not in KEEP],
    num_proc=4, desc="val",
)

print(f"Train prepared : {len(train_prep):,}")
print(f"Val prepared   : {len(val_prep):,}")

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding=self.padding, return_tensors="pt"
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids  = np.argmax(pred.predictions, axis=-1)
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)
    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
    }

print("Collator and metrics ready.")

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=ATTENTION_DROPOUT,
    hidden_dropout=HIDDEN_DROPOUT,
    feat_proj_dropout=FEAT_PROJ_DROPOUT,
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=10,
    mask_feature_prob=MASK_FEAT_PROB,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=processor.tokenizer.vocab_size,
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params : {trainable:,}")
print(f"mask_feature_prob: {model.config.mask_feature_prob}  (must be 0.1)")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP,
    lr_scheduler_type="cosine",
    fp16=FP16,
    dataloader_num_workers=4,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_prep,
    eval_dataset=val_prep,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOP_PATIENCE,
            early_stopping_threshold=EARLY_STOP_THRESHOLD,
        )
    ],
)

print(f"Starting training: {RUN_TAG}")
print(f"  Train samples : {len(train_prep):,}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
trainer.train()

FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model saved: {FINAL_DIR}")

In [ ]:
# ── Greedy WER on internal validation set ──────────────────────
val_metrics   = trainer.evaluate()
greedy_wer_val = val_metrics.get("eval_wer", float("nan"))
greedy_cer_val = val_metrics.get("eval_cer", float("nan"))
print(f"Val greedy WER : {greedy_wer_val:.4f}")

# ── KenLM on internal validation set ───────────────────────────
KENLM_BIN = "./kenlm/build/bin/lmplz"
ARPA_PATH = f"./telugu_kenlm_{RUN_TAG}.arpa"
LM_TEXT   = f"./telugu_lm_{RUN_TAG}.txt"
VOCAB_JSON = "./vocab_v2.json" if Path("./vocab_v2.json").exists() else "./vocab.json"

kenlm_wer_val = float("nan")
kenlm_wer_fleurs = float("nan")

if Path(KENLM_BIN).exists():
    import pyctcdecode

    # Build LM from train transcripts
    with open(LM_TEXT, "w", encoding="utf-8") as f:
        f.write("\n".join(train_ds["transcription"]))
    with open(LM_TEXT, "rb") as fin, open(ARPA_PATH, "w") as fout:
        subprocess.run([KENLM_BIN, "-o", "5", "--discount_fallback"],
                       stdin=fin, stdout=fout, check=True)

    with open(VOCAB_JSON) as f:
        vocab_dict = json.load(f)
    vocab_list = sorted(vocab_dict.keys(), key=lambda k: vocab_dict[k])
    decoder = pyctcdecode.build_ctcdecoder(vocab_list, ARPA_PATH, alpha=0.5, beta=1.0)
    print("KenLM decoder ready.")

    inference_model  = trainer.model.eval()
    inference_device = next(inference_model.parameters()).device

    def decode_with_kenlm(dataset):
        preds, refs = [], []
        for i, sample in enumerate(dataset):
            inp = processor(
                sample["audio"]["array"], sampling_rate=16_000,
                return_tensors="pt", padding=True,
            )
            with torch.no_grad():
                logits = inference_model(
                    input_values=inp.input_values.to(inference_device)
                ).logits[0].cpu().numpy()
            preds.append(decoder.decode(logits))
            refs.append(sample["transcription"])
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(dataset)} decoded")
        return wer_metric.compute(predictions=preds, references=refs)

    print(f"Decoding val set ({len(val_ds):,} samples) with KenLM...")
    kenlm_wer_val = decode_with_kenlm(val_ds)
    print(f"Val KenLM WER : {kenlm_wer_val:.4f}")

    # FLEURS te_in benchmark (if available)
    if fleurs_test is not None:
        print(f"Decoding FLEURS te_in test ({len(fleurs_test):,} samples) with KenLM...")
        kenlm_wer_fleurs = decode_with_kenlm(fleurs_test)
        print(f"FLEURS KenLM WER : {kenlm_wer_fleurs:.4f}")
else:
    print(f"[WARN] KenLM binary not found at {KENLM_BIN} — skipping LM decoding.")
    print("Build it: cd kenlm && mkdir build && cd build && cmake .. && make -j4")

In [ ]:
# ── Save structured results for the learning curve plot ────────
results = {
    "run_tag":           RUN_TAG,
    "dataset_source":    DATASET_SOURCE,
    "data_fraction":     DATA_FRACTION,
    "n_train_samples":   len(train_ds),
    "n_val_samples":     len(val_ds),
    "greedy_wer_val":    round(greedy_wer_val, 4),
    "greedy_cer_val":    round(greedy_cer_val, 4),
    "kenlm_wer_val":     round(kenlm_wer_val, 4),
    "kenlm_wer_fleurs":  round(kenlm_wer_fleurs, 4),
    "model":             MODEL_NAME,
    "lr":                LR,
    "effective_batch":   BATCH_SIZE * GRAD_ACCUM,
    "mask_feat_prob":    MASK_FEAT_PROB,
}

out_path = f"{RESULTS_DIR}/wav2vec2_lc_{RUN_TAG}.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print(f"RUN: {RUN_TAG}")
print(f"  Train samples    : {len(train_ds):,}")
print(f"  Greedy WER (val) : {greedy_wer_val:.4f}  ({greedy_wer_val*100:.2f}%)")
print(f"  KenLM  WER (val) : {kenlm_wer_val:.4f}  ({kenlm_wer_val*100:.2f}%)")
print(f"  KenLM  WER (FLEURS te_in) : {kenlm_wer_fleurs:.4f}  ({kenlm_wer_fleurs*100:.2f}%)")
print(f"  Results saved    : {out_path}")
print("=" * 60)
print()
print("Next: change DATA_FRACTION and re-run from Kernel → Restart & Run All")